In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

import mapper as mp

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(0)

In [ ]:
# noisy circle
n = 800
t = rng.uniform(0, 2*np.pi, n)
X = np.column_stack([np.cos(t), np.sin(t)]) + rng.normal(0, 0.03, (n, 2))

p = X[np.argmin(X[:, 0])]                   # leftmost point (kept for later cells)
f = X[:, 1]                                 # the filter values: height  f(x)=y

fig, ax = plt.subplots(figsize=(4.5, 4.5))
sc = ax.scatter(X[:, 0], X[:, 1], c=f, cmap="jet", s=12)
plt.colorbar(sc, label="filter (height)")
ax.set_aspect("equal")
plt.show()

In [ ]:
cover = mp.IntervalCover(n_intervals=8, overlap=0.4).fit(f.reshape(-1, 1))
bounds = cover.bounds_
print("interval bounds (left, right):")
for b in bounds: print("  ", np.round(b, 3))

fig, ax = plt.subplots(figsize=(7, 2.4))
ax.scatter(f, np.zeros_like(f), c=f, cmap="jet", s=8)
for i, (lo, hi) in enumerate(bounds):
    ax.add_patch(plt.Rectangle((lo, -0.4 - 0.12*i), hi-lo, 0.1,
                 alpha=0.5, color=plt.cm.tab10(i)))
ax.set_yticks([]); ax.set_xlabel("filter value")
ax.set_title("Overlapping intervals covering the range of f")
plt.show()

In [ ]:
# Pull back intervals and cluster each window
windows = cover.assign(f.reshape(-1, 1))
D = mp.euclidean_distances(X)
clusterer = mp.dbscan(eps=0.25, min_samples=5) #DBSCAN worked better for us

fig, axes = plt.subplots(1, len(windows), figsize=(2.0*len(windows), 2.2))
for ax, w, (lo, hi) in zip(axes, windows, bounds):
    ax.scatter(X[:, 0], X[:, 1], c="0.85", s=4)
    labels = clusterer(w, D)
    for L in np.unique(labels):
        m = w[labels == L]
        ax.scatter(X[m, 0], X[m, 1], c=f[m], cmap="jet",
                   vmin=f.min(), vmax=f.max(), s=8)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"[{lo:.2f},{hi:.2f}]", fontsize=8)
plt.show()

In [ ]:
# Connect clusters that share points 
m = mp.Mapper(lens=mp.Lens(mp.coordinate(1)),          # height filter f(x)=y
              cover=mp.IntervalCover(8, 0.4),
              clusterer=mp.dbscan(eps=0.25, min_samples=5)).fit(X, D)

fig, ax = plt.subplots(figsize=(5, 5))
mp.overlay_on_points(m, X, ax=ax, color_by="filter_mean", size_scale=8)
ax.set_title("Mapper graph over the data")
plt.show()
print(f"Betti = {list(m.betti_numbers())[:2]}")

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

Dc = mp.euclidean_distances(X)
filters = {
    "height": mp.coordinate(0),
    "distance from point":  mp.distance_from_point(p),
    "laplacian eigenvector": mp.graph_laplacian(1),
}

def explore(filt="height  f(x)=y", n_intervals=6, overlap=0.35):
    mm = mp.Mapper(mp.Lens(filters[filt]),
                   mp.IntervalCover(n_intervals, overlap),
                   mp.dbscan(eps=0.25, min_samples=5)).fit(X, Dc)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(X[:, 0], X[:, 1], c="0.85", s=5)
    mp.overlay_on_points(mm, X, ax=ax, color_by="filter_mean", size_scale=8)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"{filt}  —  Betti {list(mm.betti_numbers())[:2]}")
    plt.show()

interact(explore,
         filt=Dropdown(options=list(filters), value="height", description="filter"),
         n_intervals=IntSlider(value=6, min=3, max=20, step=1),
         overlap=FloatSlider(value=0.35, min=0.1, max=0.6, step=0.05));

In [ ]:
blob = rng.normal(0, 1, (1500, 2))
for Cover, name in [(mp.RectangleCover((5,5), 0.3), "rectangles"),
                    (mp.HexagonCover(5, 0.3), "hexagons")]:
    Cover.fit(blob); wins = Cover.assign(blob)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.scatter(blob[:,0], blob[:,1], c="0.85", s=5)
    for w in wins[:60]:
        if len(w): ax.scatter(blob[w,0], blob[w,1], s=4, alpha=0.3)
    ax.set_aspect("equal"); ax.set_title(f"2-D cover: {name}"); ax.axis("off")
    plt.show()

In [ ]:
def make_torus(n=2000, R=2.5, r=1.0, seed=0):
    rr = np.random.default_rng(seed)
    u = rr.uniform(0, 2*np.pi, n); v = rr.uniform(0, 2*np.pi, n)
    X3 = np.column_stack([(R+r*np.cos(v))*np.cos(u), (R+r*np.cos(v))*np.sin(u), r*np.sin(v)])
    return X3, u, v

X3, u, v = make_torus()
# embed in R^5 with a random rotation 
Xt = np.hstack([X3, np.zeros((len(X3), 2))])
Q, _ = np.linalg.qr(np.random.default_rng(1).normal(size=(5,5)))
Xt = Xt @ Q
Dt = mp.euclidean_distances(Xt)

evecs = mp.graph_laplacian(2)(Xt, Dt)          # first 2 non-trivial eigenfunctions
fig = plt.figure(figsize=(9,4))
a0 = fig.add_subplot(1,1,1, projection="3d")
a0.set_box_aspect(np.ptp(X3, axis=0))
a0.scatter(X3[:,0], X3[:,1], X3[:,2], c=evecs[:,0], cmap="jet", s=4)
a0.set_title("torus coloured by Laplacian eigenfunction 1")
plt.show()

In [ ]:
mt = mp.Mapper(mp.Lens([mp.precomputed(u), mp.precomputed(v)]),
               mp.CircularCover(n_per_axis=(6, 6), overlap=0.5),
               mp.dbscan(eps=0.6, min_samples=5)).fit(Xt, Dt)
betti = list(mt.betti_numbers())[:3]
print("Betti numbers of the Mapper complex:", betti, " (torus = [1, 2, 1])")
assert betti == [1, 2, 1]

# draw the complex embedded in 3-D, nodes fixed at cluster centers
ax = mp.plot_complex_3d(mt, X3, color_by=1)   # colour by minor angle v
ax.set_title(f"Mapper torus (abstract graph) — Betti {betti}")
plt.show()

In [ ]:
import glob, os

def load_points(path, n_sample=5000):
    import trimesh
    mesh = trimesh.load(path, force="mesh")
    P = np.asarray(mesh.vertices)
    if len(P) > n_sample:
        P = P[np.random.default_rng(0).choice(len(P), n_sample, replace=False)]
    return P

files = sorted(glob.glob("data/elephant*.*"))
if len(files) >= 2:
    poses = [load_points(files[0]), load_points(files[1])]
    print("loaded:", files[:2])
else:
    print("no files in data/")

In [ ]:
def to_2d(P):
    if P.shape[1] == 2:
        return P
    from sklearn.decomposition import PCA
    return PCA(n_components=2).fit_transform(P)

for P in poses:
    D = mp.euclidean_distances(P)
    P2 = to_2d(P)

    # 2-D lens: eccentricity + laplacian eigenvector, rectangle cover
    m2d = mp.Mapper(mp.Lens([mp.eccentricity(1.0), mp.graph_laplacian(1)]),
                    mp.RectangleCover((10, 10), 0.4),
                    mp.single_linkage(),
                    min_cluster_size=3).fit(P, D)

    # 1-D lens: eccentricity only, interval cover
    m1d = mp.Mapper(mp.Lens([mp.graph_laplacian(1)]),
                    mp.IntervalCover(15, 0.4),
                    mp.single_linkage(),
                    min_cluster_size=3).fit(P, D)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    mp.overlay_on_points(m2d, P2, ax=ax1, point_color="0.85", point_size=4,
                         color_by="filter_mean", size_scale=1)
    ax1.set_aspect("equal"); ax1.axis("off")
    ax1.set_title(f"2-D overlay (ecc + laplacian)")

    mp.plot_graph(m1d, ax=ax2, color_by="filter_mean", size_scale=None)
    ax2.set_title(f"{len(m1d.nodes_)} nodes — abstract graph (ecc, 1-D)")
    plt.tight_layout()
    plt.show()